# Computation of Monte-Carlo load cases under the assumption of Binomial distributions per room


Load the general room type dependent demands and the specific building demands

In [ ]:
import pandas as pd

from vensys_clustering import load_yaml, compute_required_volume_flows, build_zone_pmfs, save_scenario_data_to_yaml

from vensys_clustering.load_case_sampling import count_unique_rows, sample_zone_pmfs_sobol, sampled_load_cases_to_dict

from importlib.resources import files

standard_data = load_yaml(files("vensys_clustering.data").joinpath("general.yml"))
gpz_data = load_yaml("input_files/buildingA.yml")
rooms_to_merge = gpz_data["rooms_to_merge"]

compute volume flows

In [ ]:
df = compute_required_volume_flows(standard_data, gpz_data, overview_flag=True, include_revision=False)

merge the zone's binomial distributions

In [ ]:
zone_pmfs_by_time = build_zone_pmfs(df, rooms_to_merge)

sample from the pmfs using sobol sequence with `num_samples` being the amount of samples used per hour of the day

In [ ]:
room_samples, hours_vec = sample_zone_pmfs_sobol(zone_pmfs_by_time, num_samples=128)
room_samples = {room: arr.ravel() for room, arr in room_samples.items()}

count duplicates and merge them, including adding up their frequency

In [ ]:
row_counts_df = count_unique_rows(pd.DataFrame(room_samples))
row_counts_df = row_counts_df.loc[row_counts_df.drop(columns="frequency").sum(axis=1).sort_values().index].reset_index(drop=True) # sort ascending with sum of volume flow

bring in the correct format and save

In [ ]:
result_dict_sobol = sampled_load_cases_to_dict(row_counts_df)

In [ ]:
save_scenario_data_to_yaml(result_dict_sobol, "output_files/buildingA_monte_carlo_load_cases.yml")